In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\exoplanet_finder\\research\\notebook_trials'

In [7]:
os.chdir('../')

In [8]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\exoplanet_finder'

In [9]:
#1. entity update
from dataclasses import dataclass
import pathlib as Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_url: str
    local_data_file:Path

In [10]:
from exoplanet_finder.constant import *
from exoplanet_finder.utils.common import read_yaml, create_directories

In [11]:
# 4 Update configuration manager

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_URL,
            local_data_file = config.local_data_file,
        )

        return data_ingestion_config

In [12]:
import os
import urllib.request as request
import zipfile
from exoplanet_finder.logging import logger
from exoplanet_finder.utils.common import get_size

In [13]:
# 3. components

import os
from pathlib import Path
import urllib.request

from exoplanet_finder.logging import logger
from exoplanet_finder.utils.common import get_size


class DataIngestion:

    def __init__(self, config):
        self.config = config

    def download_files(self):
        """
        Downloads the DR24 TCE dataset from Hugging Face
        and stores it in the artifacts directory.
        """

        if not os.path.exists(self.config.local_data_file):

            try:

                logger.info(
                    f"Downloading dataset from {self.config.source_url}"
                )

                filename, headers = urllib.request.urlretrieve(
                    self.config.source_url,
                    self.config.local_data_file
                )

                logger.info(
                    f"Downloaded file size: "
                    f"{get_size(Path(self.config.local_data_file))}"
                )

            except Exception as e:
                raise e

        else:

            logger.info(
                f"File already exists with size: "
                f"{get_size(Path(self.config.local_data_file))}"
            )

In [16]:
#4. Pipeline

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_files()
except Exception as e:
    raise e

[2026-11-20 21:12:34,591: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-11-20 21:12:34,596: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-11-20 21:12:34,600: INFO: common: created directory at artifacts]
[2026-11-20 21:12:34,603: INFO: common: created directory at artifacts/data_ingestion]
[2026-11-20 21:12:34,605: INFO: 360121056: Downloading dataset from https://huggingface.co/datasets/Revanth214/exoplanet/resolve/main/q1_q17_dr24_tce_2026.11.16_11.35.38.csv]
[2026-11-20 21:12:36,162: INFO: 360121056: Downloaded file size: ~ 27347 KB]
